# Appendix 10.2: Tool Use

**Technique:** Tool use (function calling)

Give Claude typed **tools** (functions you define). When Claude decides to use one it returns a `tool_use` block instead of plain text. You execute the function locally, send back a `tool_result` block, and repeat until `stop_reason != "tool_use"`. The loop is just plain Python, so you stay in control of every execution.

**Contents**
* [Lesson: Tool Definitions, tool_choice, and the Agentic Loop](#lesson)
* [Worked Example: Calculator](#example)
* [Exercises](#exercises)
* Common Mistakes, Stretch, and Reflect
* [Example Playground](#playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Tool Definitions, tool_choice, and the Agentic Loop

### Tool definition shape

Each tool is a plain dict with three required keys:

```python
{
    "name": "my_tool",                        # snake_case identifier Claude uses in tool_use blocks
    "description": "What this tool does …",  # critical: Claude reads this to decide when to call it
    "input_schema": {                         # JSON Schema for the arguments Claude must supply
        "type": "object",
        "properties": {
            "arg_name": {"type": "string", "description": "…"},
        },
        "required": ["arg_name"],
    },
}
```

Pass a list of these as `tools=` to `client.messages.create`. Add `"strict": True` (with `"additionalProperties": False` in the schema) when you need a guarantee that Claude's arguments validate exactly against the schema.

### tool_choice

By default Claude decides whether to use a tool. The `tool_choice` parameter can change that:

* `{"type": "auto"}`: Claude decides (default)
* `{"type": "any"}`: Claude must use at least one tool
* `{"type": "tool", "name": "my_tool"}`: Claude must call this specific tool
* `{"type": "none"}`: Claude must not use any tool

**Model note:** forcing a tool call with `any` or `tool` is supported on many models but returns a **400** on the newest ones, including the course default `claude-opus-5-5`. On those models keep `auto` and steer with the prompt ("Use the calculator tool"), exactly as the examples below do. Check for a `tool_use` block rather than assuming one came back.

### The agentic loop

```
1. Send messages + tools → Claude
2. If stop_reason == "tool_use":
     a. Append the assistant content to messages
     b. For each tool_use block: run the tool locally
     c. Append a user message with tool_result blocks
     d. Send again → go to 2
3. stop_reason != "tool_use" → done; read the text response
```

Key invariants:
* **Always append the full `response.content` list** (not just the text block) as the next assistant message. That preserves the `tool_use` blocks, and any `thinking` blocks, in context.
* **Match `tool_use_id`** in each `tool_result` to the exact `id` from the `tool_use` block.
* **Never break the loop early**, since Claude may make multiple successive tool calls.

<a id="example"></a>
## Worked Example: Calculator

In [ ]:
# Worked example: a calculator tool with a manual agentic loop.
import ast
import operator

# Never eval() text that came from a model. This tiny evaluator only understands
# numbers, + - * / // % **, unary minus, and parentheses.
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def safe_eval(expression: str):
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression!r}")

    return _eval(ast.parse(expression, mode="eval"))


tools = [{
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the numeric result.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '12 * (3 + 4)'"}},
        "required": ["expression"],
    },
}]


def run_tool(name, tool_input):
    if name == "calculator":
        return str(safe_eval(tool_input["expression"]))
    return "unknown tool"


messages = [{"role": "user", "content": "What is 12 * (3 + 4)? Use the calculator tool."}]
response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=tools, messages=messages, **SAMPLING_PARAMS)

while response.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": response.content})
    results = []
    for block in response.content:
        if block.type == "tool_use":
            print(f"Claude called {block.name}({block.input})")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": run_tool(block.name, block.input)})
    messages.append({"role": "user", "content": results})
    response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=tools, messages=messages, **SAMPLING_PARAMS)

print(response_text(response))

<a id="exercises"></a>
## Exercises

### Exercise 10.2.1: Query a mock service

**Scenario**: Claude needs live operational data it doesn't have. You provide a `get_service_status(service)` tool backed by a mock `STATUS` dict defined in the cell. Claude calls the tool to look up the current status, then answers the question using the returned value.

**Task**: fill in the `tools` list by defining a `get_service_status` tool whose `input_schema` accepts a `service` string. The `run_tool` function and the agentic loop are already written for you.

**Success criteria**: after running the loop, the final assistant text contains the mock status value `"degraded"` for the billing service.

In [ ]:
from lib.grading import includes_any, grade

STATUS = {"billing": "degraded", "auth": "operational", "search": "operational"}

tools = [
    # [Replace: define a get_service_status tool with a `service` string input]
]

def run_tool(name, tool_input):
    if name == "get_service_status":
        return STATUS.get(tool_input["service"], "unknown")
    return "unknown tool"

messages = [{"role": "user", "content": "Is the billing service up? Use the tool, then tell me its status."}]
response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=tools, messages=messages, **SAMPLING_PARAMS)
while response.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": response.content})
    results = []
    for block in response.content:
        if block.type == "tool_use":
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": run_tool(block.name, block.input)})
    messages.append({"role": "user", "content": results})
    response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=tools, messages=messages, **SAMPLING_PARAMS)

final_text = response_text(response)

def grade_exercise(text):
    return includes_any(text, ["degraded"])

grade(final_text, grade_exercise(final_text))

In [ ]:
from hints import exercise_10_2_1_hint
print(exercise_10_2_1_hint)

<a id="common"></a>
## Common Mistakes, Stretch, and Reflect

### Common mistakes

**Forgetting to append the assistant `content` before the `tool_result`**: the API requires the full assistant turn (which includes the `tool_use` block) to appear in the message history immediately before the corresponding `tool_result` user turn. Skipping this step causes a validation error.

**Mismatched `tool_use_id`**: each `tool_result` block must carry the exact `id` from the `tool_use` block that triggered it. Copy it verbatim. Don't reconstruct or guess it.

**Not looping**: Claude may decide to call multiple tools in sequence or call a tool more than once before giving a final text answer. Calling the API only once misses every turn after the first. Always loop on `stop_reason == "tool_use"`.

**Swallowing tool failures**: if your function raises, don't drop the result. Return a `tool_result` with the error message and `"is_error": True` so Claude can recover or explain the failure.

### Stretch: the SDK tool runner

Writing the agentic loop by hand is instructive but repetitive. The Python SDK ships a beta helper that automates it: decorate a typed function with `@beta_tool` (the JSON Schema is generated from the signature and docstring) and hand it to `client.beta.messages.tool_runner`. Run the cell below to see the same calculator with no hand-written loop.

In [ ]:
# Stretch: the SDK's tool runner drives the whole loop for you.
from anthropic import beta_tool


@beta_tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the numeric result.

    Args:
        expression: An arithmetic expression, e.g. '12 * (3 + 4)'.
    """
    return str(safe_eval(expression))  # safe_eval is defined in the worked example above


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    tools=[calculator],
    messages=[{"role": "user", "content": "What is 12 * (3 + 4)? Use the calculator tool."}],
    **SAMPLING_PARAMS,
)
final_message = runner.until_done()  # runs tool calls until Claude gives a final answer
print(response_text(final_message))

`@beta_tool` + `client.beta.messages.tool_runner` handle the loop, tool dispatch, and result injection for you. Use the manual loop while learning; reach for the tool runner in production code.

### Reflect

**When do you promote a bash command to a dedicated typed tool?**

A bash command is convenient for occasional ops, but a typed tool is worth the overhead when:

* **The operation needs validation**: a typed `input_schema` catches bad arguments before they hit your infrastructure.
* **Claude needs to select from multiple capabilities**: named tools let Claude reason about which action to take, rather than constructing a freeform shell string.
* **You want auditability**: typed tool calls are structured and loggable; freeform shell strings are not.
* **The operation is dangerous**: wrapping a destructive action in a tool lets you add confirmation logic or rate limiting between the tool call and the actual execution.

<a id="playground"></a>
## Example Playground

Use the cell below to experiment with your own tool definitions. Try adding a second tool (e.g., `get_current_time` or `lookup_user`) and watch how Claude decides which to call based on the question.

In [ ]:
# Build your own tool use here.
# Suggested ideas:
#   * A unit converter tool (input: value + from_unit + to_unit)
#   * A mock database lookup tool (input: table + id)
#   * A string formatter tool (input: text + format e.g. "upper" | "snake" | "kebab")

my_tools = [{
    "name": "to_upper_case",
    "description": "Convert a string to upper case.",
    "input_schema": {
        "type": "object",
        "properties": {"text": {"type": "string", "description": "The string to convert."}},
        "required": ["text"],
    },
}]


def my_run_tool(name, tool_input):
    if name == "to_upper_case":
        return tool_input["text"].upper()
    return "unknown tool"


my_messages = [{"role": "user", "content": "Convert the phrase 'hello world' to upper case using the tool."}]
my_response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=my_tools, messages=my_messages, **SAMPLING_PARAMS)

while my_response.stop_reason == "tool_use":
    my_messages.append({"role": "assistant", "content": my_response.content})
    my_results = []
    for block in my_response.content:
        if block.type == "tool_use":
            my_results.append({"type": "tool_result", "tool_use_id": block.id, "content": my_run_tool(block.name, block.input)})
    my_messages.append({"role": "user", "content": my_results})
    my_response = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, tools=my_tools, messages=my_messages, **SAMPLING_PARAMS)

print(response_text(my_response))